# Garmin Connect Personal API: Historical Performance & Activity Analysis

This notebook demonstrates how to query Garmin Connect using the official Personal API, analyze training volume, model Fitness/Fatigue (CTL/ATL/TSB), compute Aerobic Decoupling (Cardiac Drift), and correlate recovery metrics (Sleep & HRV) with workout readiness.

In [ ]:
from datetime import date, timedelta
import pandas as pd

from ricelo.client import default_client
from ricelo.models import Activity
from ricelo.performance import aggregate_volume, calculate_fitness_fatigue
from ricelo.recovery import evaluate_daily_readiness

## 1. Fetch Recent Activities

In [ ]:
# Fetch last 60 days of activities
start_date = (date.today() - timedelta(days=60)).isoformat()
raw_activities = default_client.search_activities(start_date=start_date, limit=100)
activities = [Activity.from_api_dict(a) for a in raw_activities]

df_activities = pd.DataFrame([{
    'id': a.activity_id,
    'date': a.start_time_local[:10],
    'name': a.name,
    'type': a.activity_type,
    'distance_km': a.distance_km,
    'duration_min': round(a.duration_s / 60.0, 1),
    'pace': a.pace_min_km,
    'avg_hr': a.avg_hr,
    'aerobic_te': a.aerobic_training_effect,
    'elevation_gain_m': a.elevation_gain_m
} for a in activities])

df_activities.head(10)

## 2. Weekly Training Volume

In [ ]:
weekly_periods = aggregate_volume(activities, freq='W')
df_volume = pd.DataFrame([{
    'week': p.period_label,
    'distance_km': p.total_distance_km,
    'duration_hours': p.total_duration_hours,
    'elevation_m': p.total_elevation_m,
    'activities_count': p.total_activities
} for p in weekly_periods])

df_volume

## 3. Fitness & Fatigue Modeling (CTL, ATL, Form / TSB)

In [ ]:
fitness_records = calculate_fitness_fatigue(activities, days=60)
df_fitness = pd.DataFrame([{
    'date': r.date_str,
    'daily_load': r.daily_load,
    'ctl_fitness': r.ctl_fitness,
    'atl_fatigue': r.atl_fatigue,
    'tsb_form': r.tsb_form,
    'status': r.status
} for r in fitness_records])

df_fitness.tail(10)

## 4. Recovery & Training Readiness Assessment

In [ ]:
today_str = date.today().isoformat()
yesterday_str = (date.today() - timedelta(days=1)).isoformat()

sleep = default_client.get_sleep_summaries(yesterday_str, today_str)
hrv = default_client.get_hrv_summaries(yesterday_str, today_str)
bb = default_client.get_body_battery(yesterday_str, today_str)
daily = default_client.get_daily_summaries(today_str, today_str)

readiness = evaluate_daily_readiness(
    sleep_data=sleep[-1] if sleep else None,
    hrv_data=hrv[-1] if hrv else None,
    body_battery_data=bb[-1] if bb else None,
    daily_summary=daily[0] if daily else None
)

print(f"Verdict: {readiness.readiness_verdict}")
print(f"Recommendation: {readiness.training_recommendation}")
print(f"Sleep Score: {readiness.sleep_score} | HRV Status: {readiness.hrv_status}")